In [1]:
!pip install -q praat-parselmouth speechbrain soundfile
!git clone -q https://github.com/zehuiwu/SpeechCueLLM.git

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.1/782.1 kB 20.2 MB/s eta 0:00:00


In [2]:
import sys, torch
sys.path.append("/kaggle/working/SpeechCueLLM/feature_utils")
try:
    from speechbrain.inference.enhancement import SpectralMaskEnhancement
except ImportError:
    from speechbrain.pretrained import SpectralMaskEnhancement

enh = SpectralMaskEnhancement.from_hparams(
    source="speechbrain/metricgan-plus-voicebank",
    savedir="/kaggle/temp/mgan", run_opts={"device": "cpu"})
print("denoiser loaded")

hyperparams.yaml:   0%|          | 0.00/1.12k [00:00<?, ?B/s]

enhance_model.ckpt:   0%|          | 0.00/7.59M [00:00<?, ?B/s]

denoiser loaded


B3 Test on the file

b3.1 list the files

In [3]:
import glob, os, subprocess, soundfile as sf, torch
MELD = "/kaggle/input/datasets/zaber666/meld-dataset/MELD-RAW/MELD.Raw"
IEM  = "/kaggle/input/datasets/jiten597/iemocap/IEMOCAP_full_release"
ok = lambda f: not os.path.basename(f).startswith("._")
meld_train = sorted(filter(ok, glob.glob(f"{MELD}/train/train_splits/*.mp4")))
iem = sorted(filter(ok, glob.glob(f"{IEM}/Session*/sentences/wav/*/*.wav")))
print(len(meld_train), len(iem))   # expect about 9989 and about 10000

9989 10039


In [4]:
import sys; sys.path.append("/kaggle/working/SpeechCueLLM/feature_utils")
from extract_audio_feature import extract_audio_features_with_praat as feats

os.makedirs("/kaggle/working/tmp", exist_ok=True)
raw, den = "/kaggle/working/tmp/raw.wav", "/kaggle/working/tmp/den.wav"
subprocess.run(["ffmpeg","-y","-loglevel","error","-i",meld_train[0],
                "-vn","-ar","16000","-ac","1",raw], check=True)
out = enh.enhance_batch(enh.load_audio(raw).unsqueeze(0), lengths=torch.tensor([1.0]))
sf.write(den, out.squeeze().cpu().numpy(), 16000)
print("raw     :", feats(raw))
print("denoised:", feats(den))

raw     : {'filename': 'raw.wav', 'duration': 5.6746875, 'avg_intensity': 68.60558196069756, 'intensity_variation': np.float64(7.885954033141034), 'avg_pitch': 123.79527987029806, 'pitch_std': np.float64(21.79095302170499), 'pitch_range': np.float64(110.40460912182769), 'articulation_rate': 4.053086623712759, 'mean_hnr': 4.668370978291912}
denoised: {'filename': 'den.wav', 'duration': 5.6746875, 'avg_intensity': 75.95384683938201, 'intensity_variation': np.float64(13.395011804412402), 'avg_pitch': 148.60468950671975, 'pitch_std': np.float64(84.24447629473924), 'pitch_range': np.float64(403.0325904369142), 'articulation_rate': 4.202232435981615, 'mean_hnr': 7.421655294089826}


In [5]:
import random, time, numpy as np, pandas as pd
from tqdm import tqdm

def to_wav(src, dst):
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",src,"-vn","-ar","16000","-ac","1",dst], check=True)

def rms_db(path):
    x, _ = sf.read(path); return float(20*np.log10(np.sqrt(np.mean(x**2)) + 1e-9))

def run(sample, name):
    rows = []
    for i, src in enumerate(tqdm(sample, desc=name)):
        try:
            raw = f"/kaggle/working/tmp/{name}_raw.wav"; den = f"/kaggle/working/tmp/{name}_den.wav"
            to_wav(src, raw)
            t = time.time()
            out = enh.enhance_batch(enh.load_audio(raw).unsqueeze(0), lengths=torch.tensor([1.0]))
            sf.write(den, out.squeeze().cpu().numpy(), 16000)
            dt = time.time() - t
            r, d = feats(raw), feats(den)
            row = {"file": os.path.basename(src), "denoise_sec": dt, "dur": r["duration"],
                   "rms_raw": rms_db(raw), "rms_den": rms_db(den)}
            row.update({f"raw_{k}": v for k, v in r.items() if k not in ("filename","duration")})
            row.update({f"den_{k}": v for k, v in d.items() if k not in ("filename","duration")})
            rows.append(row)
        except Exception as e:
            tqdm.write(f"skip {src}: {e}")
    return pd.DataFrame(rows)

In [6]:
random.seed(0)
t0 = time.time(); test = run(random.sample(meld_train, 5), "t"); print((time.time()-t0)/5, "sec per clip")
print(test.iloc[:, :8])

t: 100%|██████████| 5/5 [00:02<00:00,  1.72it/s]

0.5843525886535644 sec per clip
               file  denoise_sec       dur    rms_raw    rms_den  \
0  dia652_utt12.mp4     0.057331  2.922687 -31.527477 -24.276811   
1   dia703_utt0.mp4     0.131260  7.722688 -34.645910 -27.369561   
2   dia124_utt4.mp4     0.102395  6.058687 -30.363947 -15.898739   
3   dia468_utt6.mp4     0.081436  4.565313 -22.785542 -17.745105   
4  dia851_utt18.mp4     0.047521  2.602688 -30.508196 -18.000702   

   raw_avg_intensity  raw_intensity_variation  raw_avg_pitch  
0          62.411345                 6.045556     179.265749  
1          59.359453                 7.202246     263.815889  
2          63.658116                 9.640128     157.999250  
3          71.250750                 7.179628     228.052232  
4          63.563326                 6.735229     145.322253  


In [7]:
N = 1000
random.seed(0)
meld_df = run(random.sample(meld_train, N), "meld"); meld_df.to_csv("/kaggle/working/meld_raw_vs_den.csv", index=False)
iem_df  = run(random.sample(iem, N), "iemocap");    iem_df.to_csv("/kaggle/working/iemocap_raw_vs_den.csv", index=False)
print(len(meld_df), len(iem_df))

meld:   3%|▎         | 28/1000 [00:12<08:37,  1.88it/s]/kaggle/working/SpeechCueLLM/feature_utils/syllable_nuclei.py:97: PraatWarning: The loudest and softest part in your sound differ by only 9.811278216537985 dB.
  textgrid = call(intensity, "To TextGrid (silences)", threshold3, minpause, 0.1, "silent", "sounding")
meld:   9%|▉         | 88/1000 [00:41<06:08,  2.47it/s]/kaggle/working/SpeechCueLLM/feature_utils/syllable_nuclei.py:97: PraatWarning: The loudest and softest part in your sound differ by only 9.300824679239199 dB.
  textgrid = call(intensity, "To TextGrid (silences)", threshold3, minpause, 0.1, "silent", "sounding")
meld:  18%|█▊        | 184/1000 [01:24<06:28,  2.10it/s]/kaggle/working/SpeechCueLLM/feature_utils/syllable_nuclei.py:97: PraatWarning: The loudest and softest part in your sound differ by only 6.233282422121903 dB.
  textgrid = call(intensity, "To TextGrid (silences)", threshold3, minpause, 0.1, "silent", "sounding")
meld:  23%|██▎       | 231/1000 [01:44<04:

1000 1000
